# Image -> 3D on Colab (TripoSG, free T4)

Turns a **single front image** into a **volumetric, watertight** 3D mesh using
[TripoSG](https://github.com/VAST-AI-Research/TripoSG) on a free Colab T4.

**Why TripoSG (vs SF3D):** TripoSG is a 3D shape **diffusion** model — it denoises a 3D latent,
decodes a signed-distance field, and marching-cubes it into a **closed 360 solid**. So it gives
real depth/volume from one front photo, not SF3D's flat front relief. Geometry-only output
(no texture) — bake the texture in Blender afterward (the repo already does this for cube3d).

### How to run
1. **Runtime -> Change runtime type -> T4 GPU**, then **Save**.
2. **Runtime -> Run all.** Upload your image when cell 3 prompts; a `.glb` downloads at the end.

SF3D needs numpy<2 and a CUDA op; we isolate everything in a `virtualenv` and run as a subprocess,
so Colab's base kernel is never touched (no restart). First run ~4-6 min.

> Licensing note: TripoSG's code + weights are **MIT**, but its marching-cubes op **`diso` is
> CC-BY-NC (non-commercial)**. Fine for personal/portfolio use; resolve `diso` before selling meshes.

## 1. Check the GPU (must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Install TripoSG (isolated venv)

Builds an isolated `virtualenv` with TripoSG's numpy<2 stack + its own torch, compiles the `diso`
CUDA marching-cubes op, and installs `rembg` for resale-clean background removal. Ends by printing
`tsg venv OK | numpy 1.26.x | cuda True` — that's the green light. No restart needed.

In [ ]:
import os
%cd /content
![ -d TripoSG ] || git clone https://github.com/VAST-AI-Research/TripoSG.git /content/TripoSG

# Isolated venv (virtualenv -- Colab's `python -m venv` ensurepip is broken).
!pip install -q virtualenv
!virtualenv -q --python=python3 /content/tsg-venv
PY = "/content/tsg-venv/bin/python"
!{PY} -m pip install -q -U pip wheel setuptools
# torch FIRST (cu121, T4 sm_75, fp16) so diso's CUDA extension compiles against it.
!{PY} -m pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
# requirements pins numpy==1.22.3 (no wheel, won't build) -> use the <2 ABI-match 1.26.4.
!sed -i 's/^numpy==1.22.3/numpy==1.26.4/' /content/TripoSG/requirements.txt
# diso is sdist-only -> compiles a CUDA op from source; build isolation OFF + CUDA_HOME set.
os.environ['CUDA_HOME'] = '/usr/local/cuda'
!CUDA_HOME=/usr/local/cuda {PY} -m pip install -q --no-build-isolation -r /content/TripoSG/requirements.txt
# TripoSG leaves the whole HF stack UNPINNED, so pip grabs 2025 builds that need torch 2.6+
# (transformers wants torch.float8_e8m0fnu; new diffusers/peft expect a newer huggingface_hub).
# We're on torch 2.4 (required for diso), so pin a coherent mid-2024 / torch-2.4 set.
!{PY} -m pip install -q "diffusers==0.30.0" "transformers==4.44.2" "huggingface_hub==0.24.6" "accelerate==0.33.0" "peft==0.12.0"
# rembg: 2.0.50 has no py3.12 wheel and 2.0.76 needs numpy>=2.3; 2.0.59 supports py3.12 + numpy<2.
# Also pin opencv (rembg imports cv2) to a numpy<2 build, else cv2 ABI-clashes with numpy 1.26 at run time.
!{PY} -m pip install -q "rembg==2.0.59" "opencv-python-headless==4.10.0.84" onnxruntime pillow
# Make sure nothing bumped numpy back to 2.x (diso was COMPILED against 1.26.4 -> keep that ABI).
!{PY} -m pip install -q "numpy==1.26.4"
# Sanity: diso built, GPU visible, TripoSG + rembg import. MUST print "tsg venv OK ... cuda True".
!cd /content/TripoSG && {PY} -c "import sys; sys.path.insert(0,'.'); import torch, diso, numpy, cv2, rembg; from triposg.pipelines.pipeline_triposg import TripoSGPipeline; print('tsg venv OK | numpy', numpy.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# opencv fix: rembg imports cv2, and pip may have pulled a numpy-2 opencv that clashes with
# our numpy 1.26 at run time. This pins the numpy<2 build. (On a fresh Run-all the install cell
# above already does this -- but run this cell on its own if your venv was built earlier.)
!/content/tsg-venv/bin/pip install -q "opencv-python-headless==4.10.0.84"
!/content/tsg-venv/bin/python -c "import cv2, numpy; print('cv2', cv2.__version__, '| numpy', numpy.__version__, 'OK')"

## 3. Upload your image

A clean subject works best. Run the cell and pick your file (we remove the background with rembg).

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
IMAGE_PATH = '/content/input' + os.path.splitext(next(iter(up)))[1]
shutil.move(next(iter(up)), IMAGE_PATH)
print('using', IMAGE_PATH)

## 4. Generate the mesh

Runs TripoSG **inside the venv** (subprocess, fp16) -> a watertight GLB. `STEPS`/`GUIDANCE` are the
quality knobs (defaults 50 / 7.0). Geometry only; texture is baked later in Blender.

In [ ]:
import os, glob, subprocess
PY = "/content/tsg-venv/bin/python"
STEPS = 50
GUIDANCE = 7.0

try:
    IMAGE_PATH
except NameError:
    _i = sorted(glob.glob('/content/input*'))
    assert _i, "No image found -- run the Upload cell (3) first."
    IMAGE_PATH = _i[0]
print("input:", IMAGE_PATH)

runner = r'''
import sys, os, traceback
sys.path.insert(0, "/content/TripoSG"); sys.path.insert(0, "/content/TripoSG/scripts")
os.chdir("/content/TripoSG")
try:
    import numpy as np, torch, trimesh
    from PIL import Image
    from rembg import remove, new_session
    from triposg.pipelines.pipeline_triposg import TripoSGPipeline
    from image_process import prepare_image
    from huggingface_hub import snapshot_download
    ip, out_path, steps, guidance = sys.argv[1], sys.argv[2], int(sys.argv[3]), float(sys.argv[4])
    # resale-clean bg removal -> RGBA file so prepare_image skips the non-commercial BriaRMBG path
    cut = remove(Image.open(ip).convert("RGBA"), session=new_session("u2net"))
    precut = "/content/precut.png"; cut.save(precut)
    tdir = "/content/pretrained_weights/TripoSG"
    snapshot_download("VAST-AI/TripoSG", local_dir=tdir)
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
    pipe = TripoSGPipeline.from_pretrained(tdir).to("cuda", torch.float16)
    img = prepare_image(precut, bg_color=np.array([1.0, 1.0, 1.0]), rmbg_net=None)
    out = pipe(image=img, generator=torch.Generator(device="cuda").manual_seed(42),
               num_inference_steps=steps, guidance_scale=guidance).samples[0]
    mesh = trimesh.Trimesh(out[0].astype(np.float32), np.ascontiguousarray(out[1]))
    mesh.export(out_path)
    print("DONE tris", len(mesh.faces))
except Exception:
    traceback.print_exc(); sys.exit(1)
'''
with open("/content/run_tsg.py", "w") as f:
    f.write(runner)

ret = subprocess.run([PY, "/content/run_tsg.py", IMAGE_PATH, "/content/output.glb",
                      str(STEPS), str(GUIDANCE)], capture_output=True, text=True)
print(ret.stdout)
if ret.stderr:
    print("----- venv stderr -----\n" + ret.stderr)
GLB = "/content/output.glb"
if ret.returncode != 0 or not os.path.exists(GLB):
    tail = "\n".join((ret.stdout + "\n" + ret.stderr).strip().splitlines()[-40:])
    raise RuntimeError("[X] TripoSG generation failed -- real error:\n" + tail)
print("mesh:", GLB)

## 5. Stats + download

In [ ]:
!pip install -q trimesh
import glob, trimesh
try:
    GLB
except NameError:
    _g = sorted(glob.glob('/content/output*.glb'))
    assert _g, "Run the Generate cell (4) first."
    GLB = _g[-1]
m = trimesh.load(GLB, force='mesh')
e = sorted(m.extents)
print(f'tris: {len(m.faces):,} | verts: {len(m.vertices):,} | watertight: {m.is_watertight}')
print(f'extents: {m.extents.round(3)} | depth:width ratio: {round(e[0]/e[2], 2)} (SF3D was ~0.43 = flat)')
from google.colab import files
files.download(GLB)

## 6. Back in the repo

Drop the downloaded `.glb` into `runs/` and finish in Blender (TripoSG output is **geometry only**):

```bash
# bake a 2048 BaseColor + prep to the Roblox tri budget, then validate:
roblox-ugc autoprep runs/shark/output.glb --out runs/shark/prepped.fbx --category Hat --bake
roblox-ugc inspect runs/shark/prepped.fbx --out runs/shark/report.json
roblox-ugc validate runs/shark/report.json --target accessory --category Hat
```

- **No texture/UVs** come out of TripoSG — bake them in Blender (same as the cube3d path).
- Output is dense from marching cubes — decimate to the category tri budget in prep.
- **License:** code/weights MIT, but the `diso` marching-cubes op is **CC-BY-NC** — resolve before
  selling meshes (swap to a permissive marching-cubes, or get a commercial license).

> Note: a **TPU** can't run this — custom CUDA kernels. Use the **T4 GPU** runtime.